# 12 · Web / API / 文档工具

用 Python 与网络世界沟通，并用程序自动生成与批量产出文档。

## 学习目标

- 能用 requests 函数库送出 HTTP（hypertext transfer protocol）请求，理解 get 与 post 的差异、判读状态码（status code）、解析 json 回应。
- 理解 REST API（representational state transfer）的基本交互模式，以及密钥（API key）在验证与安全上的角色与保管原则。
- 熟悉数据编码概念，能用 base64 将二进位数据转成可放进文字传输的字符串。
- 能用 urllib.parse 组装与拆解网址，并用 mimetypes 判断文件的内容类型（MIME type）。
- 能用 subprocess 从 Python 调用外部命令行工具，取得其输出。
- 能用字符串模板生成 HTML 文档，理解 markdown 转 HTML 的概念，并创建批量自动产出文档的设计思维。

## HTTP 请求基础与 requests

HTTP（hypertext transfer protocol）是程序与网络服务沟通的共同语言：你送出一个请求（request），对方回一个回应（response）。

为什么先学它：所有网络操作的根基都是「送出请求、判读是否成功、再取出数据」这三步。

两种常见方法的差异：

| 方法 | 用途 | 参数字置 |
|---|---|---|
| get | 取数据（查找） | 放在网址查找字符串 |
| post | 送数据（创建 / 提交） | 放在请求主体 body |

回应里常用的两个出口：`text`（原始文字）与 `json()`（把 json 文字解析成 Python 对象）。判读成功与否则看状态码（status code）：2xx 成功、4xx 用户端错误（如 404 找不到）、5xx 服务器错误（如 500）。

In [ ]:
# 概念：用自造的仿真回应对象示范「送请求 -> 检查状态码 -> 取数据」三步
# 不连真实网络，改用一个假的 client，回传行为与 requests 相同的对象

class FakeResponse:
    def __init__(self, status_code, payload):
        self.status_code = status_code   # 状态码：判断成功（200）或失败
        self._payload = payload          # 内部存放要回传的数据字典

    @property
    def text(self):
        import json as _json
        return _json.dumps(self._payload, ensure_ascii=False)   # text：未解析的原始字符串

    def json(self):
        return self._payload   # json()：把回应解析成 Python 对象（这里直接给字典）

def fake_get(url):
    # 造假：依网址决定回传。已知路径给 200，其余给 404
    if url.endswith('/buildings'):
        return FakeResponse(200, {'count': 2, 'items': ['塔楼A', '塔楼B']})
    return FakeResponse(404, {'error': 'not found'})

resp = fake_get('https://example.local/buildings')   # 送出 get 请求
print('状态码:', resp.status_code)

# 注意：先确认状态码成功，再调用 json()，避免对错误回应解析出垃圾
if resp.status_code == 200:
    data = resp.json()
    print('数据笔数:', data['count'])
    print('项目:', data['items'])
else:
    print('请求失败，不解析数据')

## REST API 与密钥概念

REST API（representational state transfer）是多数对外服务提供数据的风格：每个资源有一个端点（endpoint，一个网址），你带上查找参数（query parameter）与请求标头（header）去问，服务回对应结果。

密钥（API key）是用来验证（authentication）你是谁的凭证，常放在请求标头里。

为什么要学保管原则：密钥一旦写死在代码或外泄，任何人都能冒用你的额度与权限。正确做法是从环境变量或设置档读取，不硬编（hard-code）在原代码。

形状：

```
端点 + 查找参数 -> 服务 -> 回应
标头 {"Authorization": "Bearer <密钥>"}
```

In [ ]:
# 概念：自造本机假 API，示范查找参数与密钥标头如何影响回传结果
import os

VALID_KEY = 'demo-key-123'   # 仅供示范；真实情境密钥绝不写死在代码

def fake_api(endpoint, params, headers):
    # 先验证密钥：标头没有正确 key 就回 401（未授权）
    if headers.get('X-API-Key') != VALID_KEY:
        return 401, {'error': 'unauthorized'}
    # 依查找参数回传不同结果：这里用 min_floors 过滤假数据
    buildings = [('塔楼A', 25), ('街屋B', 4), ('商办C', 18)]
    min_floors = params.get('min_floors', 0)
    hits = [name for name, floors in buildings if floors >= min_floors]
    return 200, {'endpoint': endpoint, 'result': hits}

# 技巧：密钥从环境变量取，取不到才用示范预设，避免硬编到正式码
api_key = os.environ.get('BUILDING_API_KEY', VALID_KEY)

status, body = fake_api('/buildings', {'min_floors': 10}, {'X-API-Key': api_key})
print('带正确密钥:', status, body)

status2, body2 = fake_api('/buildings', {'min_floors': 10}, {'X-API-Key': 'wrong'})
print('带错误密钥:', status2, body2)

## base64 与数据编码

很多协定（例如电子邮件、json 字段）只能传纯文字，但图片、文件等是二进位（binary）数据。编码（encoding）就是换一种表示法，把字节（bytes）转成安全可传输的文字。

base64 是最常见的一种：用 64 个可打印字符表示任意字节。

重点厘清：编码不等于加密（encryption）。base64 只是换表示法，任何人都能还原，没有保密效果；要保密得另外加密。

形状：

```
bytes --base64 编码--> 文字字符串 --base64 解码--> 原 bytes
```

In [ ]:
# 概念：把二进位数据做 base64 编码再还原，验证前后一致
import base64

# 造一段假的二进位数据当示范（中文字符串先以 utf-8 转成 bytes）
raw_bytes = '建筑模型#42'.encode('utf-8')
print('原始 bytes:', raw_bytes)

encoded = base64.b64encode(raw_bytes)        # 编码：bytes -> base64 的 bytes
encoded_text = encoded.decode('ascii')       # 转成纯文字字符串，方便放进文字传输
print('base64 文字:', encoded_text)

decoded = base64.b64decode(encoded_text)     # 解码：还原回原始 bytes
print('还原 bytes:', decoded)
print('前后一致:', decoded == raw_bytes)

# 注意：base64 可被任何人还原，不具保密性，不能拿来当加密用
print('看得懂的明码:', decoded.decode('utf-8'))

## 网址解析与 MIME 类型

网址（URL）有固定结构（协定、主机、路径、查找字符串）。手动拼字符串容易在特殊字符（中文、空白、`&`）出错，所以用 urllib.parse 来正确组装与拆解。

- `urlencode`：把参数字典编成查找字符串，并自动做百分号编码（percent-encoding）。
- `urlparse`：把一条网址拆成各部位。

MIME 类型（MIME type，例如 `text/csv`、`image/png`）告诉程序一份数据该怎么处理，是上传下载与生成文档的共同语汇。`mimetypes.guess_type` 可依扩展名猜测类型。

形状：

```
基底网址 + "?" + urlencode(参数字典)
```

In [ ]:
# 概念：用 urlencode 组查找字符串、urlparse 拆网址、mimetypes 猜类型
from urllib.parse import urlencode, urlparse, parse_qs
import mimetypes

# 造一笔查找条件（含中文，需百分号编码才安全）
params = {'district': '中正区', 'min_floors': 10}
query = urlencode(params)                          # 自动把中文与符号做 percent-encoding
base = 'https://example.local/buildings'
url = base + '?' + query
print('组好的网址:', url)

parsed = urlparse(url)                             # 拆解：取出主机、路径、查找字符串等
print('主机:', parsed.netloc, '| 路径:', parsed.path)
print('拆回字典:', parse_qs(parsed.query))         # 查找字符串再还原成字典

# 依扩展名猜内容类型：下载 csv 档时告诉浏览器这是表格数据
mime, _ = mimetypes.guess_type('buildings.csv')
print('buildings.csv 的 MIME 类型:', mime)

## 用 subprocess 调用外部工具

Python 不必重造轮子，可以直接驱动既有的命令行工具（command-line tool）并接收它的输出。这是串接系统工具的关键能力。

核心是 `subprocess.run`，把命令与参数写成一个列表（list）传入，回传对象包含：

- 回传码（return code）：0 代表成功，非 0 代表失败。
- 标准输出（stdout）：工具印出的正常结果。
- 标准错误（stderr）：工具印出的错误消息。

形状：

```
subprocess.run(["命令", "参数1", "参数2"], capture_output=True, text=True)
```

In [ ]:
# 概念：用 subprocess 调用外部工具，截取输出并检查回传码
import subprocess
import sys

# 技巧：跨平台示范就直接调用目前的 Python 解释器跑一小段程序
# 命令与参数用 list 传入，避免 shell 特殊字符解读问题
cmd = [sys.executable, '-c', "print('来自外部进程的输出')"]

result = subprocess.run(cmd, capture_output=True, text=True)   # capture_output 截取 stdout/stderr，text 以文字解码

print('回传码:', result.returncode)        # 0 表示成功
print('标准输出:', result.stdout.strip())
print('标准错误:', repr(result.stderr))

# 注意：用回传码判断成功与否，而非只看有没有输出
print('运行成功:', result.returncode == 0)

## 用程序生成文档

把数据与版型分离：先写好一份含占位符（placeholder）的模板（template），再用变量把内容填进去，就能大量产生结构一致的文档。

为什么这样设计：版型只写一次，数据换一批就重新产出一份，维护成本低。

markdown 与 HTML 的关系：markdown 是给人写的精简标记，最终常被转成 HTML 给浏览器显示；理解两者对应有助于选择输出格式。

安全意识：若数据来自外部，填入 HTML 前要做跳脱（escaping），把 `<`、`>`、`&` 换成安全写法，避免破坏版面或被注入。

In [ ]:
# 概念：用含占位符的 HTML 模板字符串，填入数据字典产出完整 HTML
from html import escape

# 造一笔假数据当示范
building = {'name': '中正塔 <示范>', 'floors': 25, 'height': 88.5}

# 模板用 {名称} 当占位符，之后用 format 填值
template = (
    '<div class="card">\n'
    '  <h2>{name}</h2>\n'
    '  <p>楼层数：{floors}</p>\n'
    '  <p>楼高：{height} 公尺</p>\n'
    '</div>'
)

# 注意：数据含 < > 等字符，填入前先 escape，避免破坏 HTML 结构
html = template.format(
    name=escape(str(building['name'])),
    floors=building['floors'],
    height=building['height'],
)
print(html)

## 自动化批量产出的设计思维

单笔能跑只是起点。把流程拆成可重复、可容错、可批量的步骤，才是自动化的核心。

几个关键概念：

- 批量处理（batch processing）：对一份数据清单逐笔套用相同流程。
- 失败处理与重试（retry）：遇到坏掉的一笔，记录并跳过或重试，不让整批中断。
- 可重复运行（idempotency）：同样输入重跑多次，结果一致、不重复产生副作用。

设计原则：把「输入数据 -> 套模板 -> 输出文档」做成一个可逐笔调用的函数，外层用循环批量驱动，并收集成功与失败清单。

In [ ]:
# 概念：批量套模板产文档，遇到问题项目记录并跳过而不中断整批
template = '<li>{name}：{floors} 层</li>'

# 造一份多笔数据，刻意混入一笔缺字段的坏数据
records = [
    {'name': '塔楼A', 'floors': 25},
    {'name': '街屋B'},                 # 缺 floors，会在套模板时出错
    {'name': '商办C', 'floors': 18},
]

outputs = []
skipped = []   # 容错：把失败项目记下来，整批跑完再回报
for i, rec in enumerate(records):
    try:
        outputs.append(template.format(**rec))   # 缺键会丢 KeyError
    except KeyError as e:
        skipped.append((i, str(e)))               # 记录哪一笔、缺什么，然后继续

print('成功产出:')
for line in outputs:
    print(' ', line)
print('被跳过的项目（索引, 原因）:', skipped)

## 练习

以下三题由浅入深，皆为集成多个概念的小任务。数据请自己用 list 造，不引用任何外部文件。只需在 `# TODO` 处完成。

In [ ]:
# TODO 1 ·（简单）建筑清单查找网址（集成：网址解析 urlencode/urlparse + MIME 类型）
#   情境：替一份建筑数据下载页组出带查找参数的网址，并标出下载文件的类型。
#   要求：
#     1. 在程序内用字典自造一笔查找条件，例如 {"district": "中正区", "min_floors": 10}，
#        用 urllib.parse.urlencode 编成查找字符串，再接上一个基底网址。
#     2. 给定下载文件名 "buildings.csv"，用 mimetypes.guess_type 查出 MIME 类型并印出。
#   验收：应看到一条把中文与数字正确百分号编码后的完整网址，以及 text/csv 这个 MIME 类型。

# TODO: 在这里完成

In [ ]:
# TODO 2 ·（中间）建筑数据抓取报表（集成：HTTP 请求 + 状态码 + json 解析 + HTML 生成）
#   情境：从一个自造的本机假 API 取回多笔建筑数据，整理成一份 HTML 报表。
#   要求：
#     1. 自写一个假 API 函数，回传含 status_code 与 json() 方法的仿真回应对象，
#        内含数笔建筑数据（每笔有 height、floors、footprint，用 list 自造仿真数字）。
#     2. 调用后先检查状态码是否为 200，成功才调用 json() 取数据；非 200 则印错误并停止。
#     3. 为每笔算出楼地板面积 GFA（gross floor area，以 footprint × floors 估算），
#        把所有字段填进一份含占位符的 HTML 模板，每笔成为表格一列。
#     4. 汇整成单一 HTML 报表字符串并印出。
#   验收：应看到一段完整 HTML，表格中每栋建筑一列，且含一栏由程序算出的 GFA。

# TODO: 在这里完成

In [ ]:
# TODO 3 ·（稍难）容积政策情境的街廓聚合比较
#   （集成：json 数据 + numpy 聚合 + 批量产出 + 文档生成 + 容错）
#   情境：给定多个街廓 block 内的建筑数据，比较容积率 FAR 政策调整前后各街廓总楼地板面积变化，
#         并输出一份摘要文档。
#   要求：
#     1. 用 list 或 numpy 自造数笔建筑数据，每笔含街廓编号 cell、footprint、floors，
#        其中刻意混入一两笔缺漏值（例如 floors 为 None）。
#     2. 设计政策情境：对符合条件的街廓套用高度乘数 multiplier（例如允许楼层数 ×1.5），
#        算出调整后 GFA；遇缺漏项目要跳过或记录而不中断整批（容错）。
#     3. 依街廓编号 cell 聚合出政策前、政策后的总 GFA，并算出每个街廓的增幅百分比。
#     4. 把各街廓的前后对照与增幅，用模板批量填成 HTML 或 markdown 摘要，
#        并在末尾列出被跳过的缺漏项目。
#   验收：应看到一份依街廓分组的对照摘要，含政策前后总 GFA 与增幅百分比，
#         且明确标示哪些缺漏项目被略过。

# TODO: 在这里完成

## 小结

- HTTP 操作的根基是三步：送出请求、用状态码（status code）判读是否成功、再用 `json()` 取出结构化数据。
- REST API 以端点加查找参数组成查找，密钥（API key）用于验证且绝不可硬编或外泄，应从环境变量或设置取得。
- base64 是编码（换表示法）而非加密，任何人都能还原，仅用于把二进位数据塞进文字信道。
- urllib.parse 负责安全组装与拆解网址（含百分号编码），mimetypes 依扩展名判断内容类型（MIME type）。
- subprocess.run 让 Python 驱动外部命令行工具，用回传码（return code）判断成败，并截取 stdout 与 stderr。
- 文档生成的核心是数据与模板分离，再以批量处理（batch processing）配合容错，把单笔流程扩展成可重复、可容错的自动化。